<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/13-performance-tuning/01-diagnosing-oom-spill-skew-partitions.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 13.1 — Symptom classifier: spill, skew, or bad partition count

Reproduce all three non-fatal symptoms in separate stages, then write
one classifier function that reads REST metrics and names which
symptom a given stage has. OOM is discussed but not triggered (it
would kill the kernel) — instead we build the SAFE GUARD pattern you'd
use to prevent one.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("13.1-diagnosing-symptoms")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
import urllib.request, json as _json

def app_id():
    return spark.sparkContext.applicationId

def stage_detail(stage_id):
    url = f"http://localhost:4040/api/v1/applications/{app_id()}/stages/{stage_id}"
    return _json.load(urllib.request.urlopen(url))[0]

def last_stage_ids(n=1):
    url = f"http://localhost:4040/api/v1/applications/{app_id()}/stages?status=complete"
    ids = sorted(s["stageId"] for s in _json.load(urllib.request.urlopen(url)))
    return ids[-n:]

## Reproduce symptom 1: spill (12.2) — too few partitions, big rows

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "2")
spill_df = (spark.range(6_000_000)
            .withColumn("v", (col("id") * 2654435761) % 1000000007)
            .withColumn("pad", F.lit("z" * 300)))
spill_df.orderBy("v").count()
spill_stage = last_stage_ids(1)[0]
print("spill demo -> stage", spill_stage)

## Reproduce symptom 2: skew (7.4/10.4) — one hot key

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
skew_df = spark.range(4_000_000).withColumn(
    "k", F.when(col("id") % 100 < 90, F.lit(0)).otherwise((col("id") % 100).cast("int")))
skew_df.groupBy("k").count().collect()
skew_stage = last_stage_ids(1)[0]
print("skew demo -> stage", skew_stage)

## Reproduce symptom 3: too many partitions — overhead dominates

In [ ]:
toomany_df = spark.range(50_000).repartition(400)   # way more partitions than rows justify
toomany_df.groupBy((col("id") % 10).alias("k")).count().collect()
toomany_stage = last_stage_ids(1)[0]
print("too-many-partitions demo -> stage", toomany_stage)

## The classifier: one function, three verdicts

Reads spill bytes (12.2) and per-task duration quantiles (13.0)
off the same stage-detail payload and names the symptom.

In [ ]:
def classify_stage(stage_id):
    detail = stage_detail(stage_id)
    tasks = list(detail["tasks"].values())
    n = len(tasks)
    durs = sorted(t["duration"] for t in tasks)
    disk_spill = sum(t["taskMetrics"]["diskBytesSpilled"] for t in tasks)
    median = durs[n // 2] if n else 0
    mx = durs[-1] if n else 0
    ratio = mx / max(median, 1)

    if disk_spill > 0 and ratio <= 5:
        verdict = "SPILL (evenly spread -> too few partitions / undersized memory)"
    elif disk_spill > 0 and ratio > 5:
        verdict = "SPILL concentrated on one task -> actually SKEW, not partition count"
    elif ratio > 5:
        verdict = "SKEW (one task dominates, no spill)"
    elif n > 100 and median < 50:
        verdict = "TOO MANY PARTITIONS (many tiny tasks, high overhead)"
    else:
        verdict = "healthy"
    print(f"stage {stage_id}: n_tasks={n} median={median}ms max={mx}ms "
          f"disk_spill={disk_spill:,}B -> {verdict}")

for sid in (spill_stage, skew_stage, toomany_stage):
    classify_stage(sid)

## The one symptom we do NOT trigger here: OOM

Forcing a real driver or executor OOM kills the Python kernel /
JVM — not something to do in a shared notebook. Instead, build the
GUARD that prevents the most common driver-OOM cause: `.collect()` on
unbounded data.

In [ ]:
def safe_collect(df, max_rows=100_000):
    """Refuses to collect() more than max_rows to the driver -- the
    13.1 guard against the most common driver-OOM cause."""
    n = df.count()
    if n > max_rows:
        raise ValueError(
            f"refusing to collect {n:,} rows (limit {max_rows:,}) -- "
            f"use .write(...) or aggregate further instead")
    return df.collect()

try:
    safe_collect(spark.range(500_000))
except ValueError as e:
    print("guard fired as intended:", e)

print("small collect still works:", len(safe_collect(spark.range(10))))

In [ ]:
spark.stop()

## Exercises

1. Run `classify_stage` on a stage from Module 12.2's original spill
   demo (or rebuild it here) and confirm it lands in the "evenly
   spread" bucket, not the skew bucket.
2. The classifier's thresholds (`ratio > 5`, `n > 100 and median <
   50`) are arbitrary. Break it: construct a stage that's genuinely
   skewed but has `ratio` just under 5. What real-world data
   distribution would produce that?
3. Extend `classify_stage` to also check shuffle read/write byte
   imbalance across tasks (not just duration) — does it agree with
   the duration-based verdict on all three demo stages?
4. Modify `safe_collect` to also work as a guard on `.toPandas()`.
   What Module 3 lesson does this generalize?
5. The lesson describes three OOM sub-types: driver, executor-during-
   shuffle, and Python-worker. `safe_collect` guards against the
   first. Sketch (in comments, no need to run) what a guard against
   the third (Python-worker OOM from a UDF) would check.

In [ ]:
# your exercise code here